### Structured Output ###

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic - Pi Dent Tak

Pydantic models provide the richest feature set with field validation, descriptions and nested structures.

In [2]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002748ED912B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002748ED91D30>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="This title of the movie")
    description: str = Field(description="This movie description")
    year: int = Field(description="This year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="the movie's rating out of 10")

In [12]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002748ED912B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002748ED91D30>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'This t

In [7]:
# This is how we get default output of movie
model.invoke("Provide details about the movie Inception")

AIMessage(content='**Inception (2010) – Quick Reference**\n\n| Item | Details |\n|------|---------|\n| **Title** | *Inception* |\n| **Release Year** | 2010 |\n| **Genre** | Science‑fiction, Action, Thriller |\n| **Running Time** | 148 minutes |\n| **Director** | Christopher Nolan |\n| **Screenwriter** | Christopher Nolan (story & screenplay) |\n| **Producers** | Emma Thomas, Christopher Nolan, Charles Roven, Jordan Goldberg |\n| **Production Companies** | Warner Bros. Pictures, Legendary Pictures, Syncopy Inc. |\n| **Distributor** | Warner Bros. Pictures |\n| **Budget** | Approximately **$160\u202fmillion** |\n| **Box‑Office Gross** | ≈\u202f$839\u202fmillion worldwide |\n| **MPAA Rating** | PG‑13 (for sequences of violence and action, some language, brief sexual content) |\n\n---\n\n### Main Cast\n\n| Actor/Actress | Character | Brief Description |\n|---------------|-----------|-------------------|\n| **Leonardo DiCaprio** | Dom Cobb | A skilled “extractor” who enters people’s dreams 

In [13]:
# This is how we get Structured output of movie
model_with_structure.invoke("Provide details about the movie Inception")

Movie(title='Inception', description="A skilled thief, Dom Cobb, who specializes in extracting secrets from deep within the subconscious during the dream state, is offered a chance to have his criminal record erased if he can successfully perform an 'inception'—planting an idea into a target's mind. He assembles a team and navigates layered dream worlds, confronting his own past and the blurred line between reality and illusion.", year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed strucutre

In [14]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """ A movie with details """
    title: str = Field(..., description="This title of the movie")
    description: str = Field(..., description="This movie description like 10 character after that ...")
    year: int = Field(..., description="This year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="the movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide details about the movie "Inception". Use the function Movie with fields: description, director, rating, title, year. Provide description limited to about 10 characters? The function comment: "This movie description like 10 character after that ..." unclear. Probably they want a short description maybe 10 characters? Might be a typo. We can give a brief description. Use function.', 'tool_calls': [{'id': 'fc_7214fe1c-261b-4d68-a481-622ed2767ee6', 'function': {'arguments': '{"description":"Dream heist thriller","director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 177, 'total_tokens': 321, 'completion_time': 0.306624018, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.093507859, 'prompt_tokens_details': None, 'queue_time': 0.355597801,

### Nested Structure

In [16]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Professor Stephen Miles'), Actor(name='Dileep Rao', role='Yusuf'), Actor(name='Pete Postlethwaite', role='Robert Fischer Sr.')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)

### TypedDict

TypeDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation